# Biomarker S7 — Phân loại KL theo thứ tự (ordinal) trên bảng biomarker

**Input:** `s6_fcl/biomarker_table_v2.csv` (S6). Nếu chưa có, dùng `biomarker_table.csv` (S3, chỉ cột legacy).
**Output (thư mục mới):** `knee_biomarkers/s7_ordinal/` — metric theo fold, dự đoán out-of-fold, hình, config.

**Câu hỏi notebook trả lời:** xử lý KL như **thứ tự** (KL0 < … < KL4) có tốt hơn 5 lớp rời rạc (S4) không, và
head ordinal của slide *Biomarker-guided ordinal multi-task* hành xử thế nào trên feature bảng.

**Thiết kế đánh giá (giống nhau cho mọi model):**
- `StratifiedGroupKFold` 5 fold **theo subject**, lặp 3 seed ⇒ 15 fold; cùng fold cho mọi model / feature set.
- Impute median **fit trên train** của từng fold.
- Metric chính **QWK** (quadratic-weighted kappa, chuẩn cho KL grading), kèm MAE và accuracy. Báo cả per-fold (mean ± sd)
  và trên dự đoán out-of-fold gộp theo seed (ổn định hơn khi lớp hiếm).

**Năm model** (định nghĩa trong `bsc/ordinal.py`, có test):

| | Model | Xử lý thứ tự | Giải mã |
|---|---|---|---|
| A | XGB softmax (S4) | không | argmax |
| B | XGB Frank & Hall: 4 bộ nhị phân P(KL>k) | có | đếm p_k > 0.5 (sau cummin) |
| C | XGB hồi quy KL + điểm cắt tối ưu QWK | có | digitize |
| D | MLP loss ngưỡng + L_mono | có | đếm p_k > 0.5 |
| E | MLP **đúng loss slide**: ngưỡng + L_mono + softmax CE + OA BCE | có | đếm p_k > 0.5 (so với head softmax) |

**Quy tắc giải mã được ghi rõ** vì slide chưa nói: `y = #{k : p_k > 0.5}`. Với ca KL3, p₁, p₂ "cao quá" không gây sai;
sai chỉ đến từ p₃ > 0.5 (lên KL4) hoặc p₂ < 0.5 (xuống KL2) — mục 5 đếm trực tiếp hai hướng này.
`L_OA` (KL ≥ 2) trùng sự kiện với ngưỡng p₁ = P(KL > 1); ở đây target OA tính từ **giá trị KL thật**, không từ chỉ số lớp
(cohort có thể thiếu KL0).

In [ ]:
# ============================================================
# Moi truong: Drive + clone repo. MOI biomarker/model co MOT dinh nghia trong bsc/*.py,
# notebook chi goi - khong copy code vao day (quy tac "mot dinh nghia" cua CLAUDE.md).
# ============================================================
!pip install -q xgboost scikit-learn pandas matplotlib tqdm 2>/dev/null
from google.colab import drive
drive.mount("/content/drive")

REPO_URL, REPO_DIR = "https://github.com/AIVIETNAM-AIO-Tuan/bsCart-net.git", "/content/repo"
import os, sys
if not os.path.isdir(f"{REPO_DIR}/bsc"):
    !git clone -q $REPO_URL $REPO_DIR
!cd $REPO_DIR && git pull -q
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
for _m in [k for k in list(sys.modules) if k == "bsc" or k.startswith("bsc.")]:
    del sys.modules[_m]

import torch
from bsc import ordinal as ORD
from bsc.biomarkers import LEGACY_PREFIXES, SURFACE_PREFIXES
print("bsc.ordinal:", ORD.__file__, "| torch", torch.__version__)

## 0) Nạp bảng, chọn feature set, mã hóa nhãn

In [ ]:
from pathlib import Path
import json, time
import numpy as np, pandas as pd
from tqdm.auto import tqdm

BIOM_DIR = Path("/content/drive/MyDrive/knee_biomarkers")
V2_CSV = BIOM_DIR / "s6_fcl" / "biomarker_table_v2.csv"     # tu S6
S3_CSV = BIOM_DIR / "biomarker_table.csv"                    # fallback: bang S3 (chi legacy)
OUT_DIR = BIOM_DIR / "s7_ordinal"
OUT_DIR.mkdir(parents=True, exist_ok=True)
assert str(OUT_DIR).startswith("/content/drive/"), "Drive-first"

SRC = V2_CSV if V2_CSV.exists() else S3_CSV
df = pd.read_csv(SRC).drop_duplicates("case_id").reset_index(drop=True)
df = df.dropna(subset=["KL", "subject"]).copy()
df["KL"] = df["KL"].astype(int)
print("nguon:", SRC.name, "| ca:", len(df), "| subject:", df["subject"].nunique())

legacy_cols = [c for c in df.columns if c.startswith(LEGACY_PREFIXES)]
surface_cols = [c for c in df.columns if c.startswith(SURFACE_PREFIXES)]
FEATURE_SETS = {"legacy_s3": legacy_cols}
if surface_cols:
    FEATURE_SETS["s6_surface_only"] = surface_cols
    FEATURE_SETS["s6_all"] = legacy_cols + surface_cols
print("so feature:", {k: len(v) for k, v in FEATURE_SETS.items()})

# Nhan: chi so lop 0..K-1 theo cac muc KL CO MAT (cohort co the thieu KL0)
CLASSES = sorted(df["KL"].unique().tolist())
K = len(CLASSES)
cls_to_idx = {c: i for i, c in enumerate(CLASSES)}
y_idx = df["KL"].map(cls_to_idx).to_numpy(np.int64)
groups = df["subject"].astype(str).to_numpy()
oa_target = (df["KL"] >= 2).to_numpy(np.float32)      # OA status = KL>=2 (slide 5/11), tu gia tri KL THAT
print("KL classes:", CLASSES, "| n moi lop:", np.bincount(y_idx, minlength=K).tolist())
print("nguong t_k = 1(KL > k):", [f"KL>{c}" for c in CLASSES[:-1]])

## 1) CV theo subject — cùng fold cho mọi model

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
N_SPLITS, SEEDS = 5, [0, 1, 2]
FOLDS = []
for s in SEEDS:
    sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=s)
    for f, (tr, te) in enumerate(sgkf.split(df, y_idx, groups)):
        assert not (set(groups[tr]) & set(groups[te])), "ro ri subject giua train/test"
        FOLDS.append((s, f, tr, te))
print(len(FOLDS), "fold | kich thuoc test:", sorted({len(te) for *_, te in FOLDS}))

## 2) Năm model — cùng dữ liệu, khác cách xử lý thứ tự

Siêu tham số XGB giữ nguyên S4 để so sánh công bằng. MLP: 2 lớp ẩn 64/32, full-batch Adam 400 epoch, không tinh chỉnh.
Mục tiêu là so *cách xử lý thứ tự*, không phải săn siêu tham số.

In [ ]:
from xgboost import XGBClassifier, XGBRegressor
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold, cross_val_predict

XGB_KW = dict(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8,
              colsample_bytree=0.8, random_state=42, n_jobs=2)   # = S4
MLP_EPOCHS = 400

def m_softmax(Xtr, ytr, Xte, **_):
    clf = XGBClassifier(objective="multi:softprob", num_class=K, eval_metric="mlogloss", **XGB_KW)
    clf.fit(Xtr, ytr)
    p = clf.predict_proba(Xte)
    return dict(y_pred=p.argmax(1), p_cls=p)

def m_frankhall(Xtr, ytr, Xte, **_):
    fh = ORD.FrankHall(lambda: XGBClassifier(objective="binary:logistic", eval_metric="logloss", **XGB_KW))
    fh.fit(Xtr, ytr, K)
    p = fh.predict_proba_thresholds(Xte)
    return dict(y_pred=ORD.decode_count(p), p_thr=p)

def m_reg_cutpoints(Xtr, ytr, Xte, gtr=None, **_):
    reg = XGBRegressor(objective="reg:squarederror", **XGB_KW)
    inner = cross_val_predict(reg, Xtr, ytr, groups=gtr, cv=GroupKFold(n_splits=4))   # diem HONEST cho diem cat
    cuts = ORD.fit_cutpoints(inner, ytr, K)
    reg.fit(Xtr, ytr)
    return dict(y_pred=ORD.apply_cutpoints(reg.predict(Xte), cuts), cuts=cuts)

def m_mlp_ordinal(Xtr, ytr, Xte, **_):
    model, info = ORD.train_ordinal_mlp(Xtr, ytr, K, lambdas=ORD.ORDINAL_ONLY_LAMBDAS, epochs=MLP_EPOCHS, seed=0)
    out = ORD.predict_ordinal_mlp(model, info, Xte)
    return dict(y_pred=out["y_count"], p_thr=out["p_thr"], y_alt=out["y_cumdiff"])

def m_mlp_slide(Xtr, ytr, Xte, oa_tr=None, **_):
    model, info = ORD.train_ordinal_mlp(Xtr, ytr, K, oa_target=oa_tr, lambdas=ORD.SLIDE_LAMBDAS,
                                        epochs=MLP_EPOCHS, seed=0)
    out = ORD.predict_ordinal_mlp(model, info, Xte)
    return dict(y_pred=out["y_count"], p_thr=out["p_thr"], y_alt=out["y_softmax"], p_cls=out["p_cls"])

MODELS = {
    "A_xgb_softmax":         m_softmax,        # nominal (S4)
    "B_xgb_frankhall":       m_frankhall,      # ordinal: K-1 nhi phan, giai ma dem
    "C_xgb_reg_cutpoints":   m_reg_cutpoints,  # ordinal: hoi quy + diem cat QWK
    "D_mlp_ordinal_only":    m_mlp_ordinal,    # ordinal: loss nguong + L_mono
    "E_mlp_slide_multitask": m_mlp_slide,      # dung loss slide: nguong + L_mono + softmax + OA
}

## 3) Chạy CV — gom dự đoán out-of-fold (~10–15 phút CPU)

In [ ]:
records, OOF = [], {}
t_start = time.time()
for fs_name, cols in FEATURE_SETS.items():
    X_all = df[cols].to_numpy(np.float32)
    for (seed, f, tr, te) in tqdm(FOLDS, desc=fs_name):
        imp = SimpleImputer(strategy="median").fit(X_all[tr])       # impute CHI tu train
        Xtr, Xte = imp.transform(X_all[tr]), imp.transform(X_all[te])
        for m_name, fn in MODELS.items():
            r = fn(Xtr, y_idx[tr], Xte, gtr=groups[tr], oa_tr=oa_target[tr])
            key = (fs_name, m_name, seed)
            st = OOF.setdefault(key, dict(y_pred=np.full(len(df), -1), y_alt=np.full(len(df), -1),
                                          p_thr=np.full((len(df), K - 1), np.nan)))
            yp = np.asarray(r["y_pred"], np.int64)
            st["y_pred"][te] = yp
            if r.get("p_thr") is not None:
                st["p_thr"][te] = r["p_thr"]
            if r.get("y_alt") is not None:
                st["y_alt"][te] = r["y_alt"]
            records.append(dict(feature_set=fs_name, model=m_name, seed=seed, fold=f, n_test=len(te),
                                qwk=ORD.qwk(y_idx[te], yp, K), acc=float((yp == y_idx[te]).mean()),
                                mae=ORD.mae(y_idx[te], yp)))
res = pd.DataFrame(records)
res.to_csv(OUT_DIR / "cv_fold_metrics.csv", index=False)
print(f"xong {len(res)} (feature_set, model, fold) trong {(time.time() - t_start) / 60:.1f} phut")

## 4) Kết quả — nominal (A) vs ordinal (B–E), theo feature set

Đọc: (1) B–E có hơn A về **QWK/MAE** không (ordinal thường được MAE, accuracy có thể không đổi); (2) thêm cột S6 có giúp
so với legacy không. Sd qua fold thường 0.05–0.1 với N vài trăm ⇒ chênh dưới mức đó chưa kết luận được.

In [ ]:
import matplotlib.pyplot as plt
pd.set_option("display.width", 180)
fold_summary = res.groupby(["feature_set", "model"])[["qwk", "acc", "mae"]].agg(["mean", "std"]).round(3)
print("per-fold (mean ± sd qua 15 fold):")
display(fold_summary)

pooled = []
for (fs, m, s), st in OOF.items():
    ok = st["y_pred"] >= 0
    pooled.append(dict(feature_set=fs, model=m, seed=s, qwk=ORD.qwk(y_idx[ok], st["y_pred"][ok], K),
                       mae=ORD.mae(y_idx[ok], st["y_pred"][ok]), acc=float((st["y_pred"][ok] == y_idx[ok]).mean())))
pooled = pd.DataFrame(pooled)
pooled_summary = pooled.groupby(["feature_set", "model"])[["qwk", "mae", "acc"]].agg(["mean", "std"]).round(3)
print("out-of-fold gop (mean ± sd qua 3 seed):")
display(pooled_summary)

fig, ax = plt.subplots(figsize=(10, 4))
piv = pooled.groupby(["model", "feature_set"])["qwk"].mean().unstack("feature_set")
piv_sd = pooled.groupby(["model", "feature_set"])["qwk"].std().unstack("feature_set")
piv.plot.bar(ax=ax, yerr=piv_sd, capsize=3, rot=15)
ax.set_ylabel("QWK out-of-fold (mean ± sd qua seed)")
ax.set_title("A = nominal (S4); B-E = ordinal")
plt.tight_layout(); plt.savefig(OUT_DIR / "qwk_by_model.png", dpi=130); plt.show()

## 5) Chẩn đoán head ordinal — bảng p_k theo KL thật, hướng sai, khớp giữa các cách giải mã

- **Bảng p_k trung bình theo KL thật:** lý tưởng là bậc thang (hàng KL3: cao, cao, cao, thấp). Ngưỡng cuối
  P(KL>3) thường hiệu chỉnh kém nhất vì KL4 hiếm.
- **Hướng sai ở lớp kề cuối (KL3):** đếm trực tiếp "sai LÊN do p₃ > 0.5" và "sai XUỐNG do p₂ ≤ 0.5".
- **Khớp giải mã:** E có head softmax và head ordinal cùng dự đoán KL; tỉ lệ bất đồng là mức "3 head mâu thuẫn" mà slide chưa xử lý.

In [ ]:
from sklearn.metrics import confusion_matrix
FS_SHOW = "s6_all" if "s6_all" in FEATURE_SETS else list(FEATURE_SETS)[0]
SEED_SHOW = SEEDS[0]
thr_names = [f"P(KL>{c})" for c in CLASSES[:-1]]
for m_name in ["B_xgb_frankhall", "D_mlp_ordinal_only", "E_mlp_slide_multitask"]:
    st = OOF[(FS_SHOW, m_name, SEED_SHOW)]
    ok = ~np.isnan(st["p_thr"]).any(axis=1)
    p, y, yp = st["p_thr"][ok], y_idx[ok], st["y_pred"][ok]
    print(f"\n==== {m_name} | {FS_SHOW} | seed {SEED_SHOW} | n={int(ok.sum())}")
    print("p_k trung binh theo KL that (ly tuong: bac thang 1..1,0..0):")
    display(pd.DataFrame(ORD.mean_p_by_class(p, y, K), index=[f"KL{c}" for c in CLASSES], columns=thr_names).round(2))
    print("AUC tung nguong:", dict(zip(thr_names, np.round(ORD.threshold_auc(p, ORD.to_thresholds(y, K)), 3))))
    print(f"hang p khong don dieu: {ORD.monotonic_violation_rate(p):.3f}")
    print(f"doan CAO hon that: {(yp > y).mean():.3f} | THAP hon that: {(yp < y).mean():.3f}")
    if K >= 3:
        c3 = K - 2                                   # lop ke cuoi (= KL3 neu du KL0..4)
        m3 = y == c3
        if m3.any():
            over = (p[m3, c3] > 0.5).mean()          # p_{c3} > 0.5  -> len lop cuoi
            under = (p[m3, c3 - 1] <= 0.5).mean()    # p_{c3-1} <= 0.5 -> xuong lop duoi
            print(f"KL{CLASSES[c3]} (n={int(m3.sum())}): sai LEN do {thr_names[c3]}>0.5: {over:.3f} | "
                  f"sai XUONG do {thr_names[c3 - 1]}<=0.5: {under:.3f}")
    alt = st["y_alt"][ok]
    if (alt >= 0).all():
        name = "head softmax" if "slide" in m_name else "cumdiff"
        print(f"khop giai ma dem vs {name}: {(alt == yp).mean():.3f}")
    print("confusion (hang = that, cot = doan; thu tu", CLASSES, ")")
    print(confusion_matrix(y, yp, labels=list(range(K))))

## 6) Sensitivity — chỉ case tin cậy cao (như S4 mục 5)

So QWK khi train/test trên toàn cohort vs chỉ case có ít nhất một phần GT thật. Lệch > 0.1 ⇒ nhãn AI đang chi phối kết luận.

In [ ]:
HIGH_CONF = {"gt_real_imorphics", "partial_gt_oaizib_ai_meniscus_patella"}
if "source_dataset" not in df.columns:
    print("bo qua: thieu cot source_dataset")
else:
    hc = df["source_dataset"].isin(HIGH_CONF).to_numpy()
    print("case tin cay cao:", int(hc.sum()), "/", len(df))
    cols = FEATURE_SETS[FS_SHOW]
    X_all = df[cols].to_numpy(np.float32)
    rows = []
    for m_name in ["A_xgb_softmax", "B_xgb_frankhall", "E_mlp_slide_multitask"]:
        for label, sel in [("toan_bo", np.ones(len(df), bool)), ("tin_cay_cao", hc)]:
            idx = np.flatnonzero(sel)
            if len(np.unique(y_idx[idx])) < 2 or len(np.unique(groups[idx])) < 10:
                print(f"[{m_name}/{label}] qua it du lieu -> bo qua")
                continue
            yp_all = np.full(len(df), -1)
            sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED_SHOW)
            for tr_, te_ in sgkf.split(idx, y_idx[idx], groups[idx]):
                tr, te = idx[tr_], idx[te_]
                imp = SimpleImputer(strategy="median").fit(X_all[tr])
                r = MODELS[m_name](imp.transform(X_all[tr]), y_idx[tr], imp.transform(X_all[te]),
                                   gtr=groups[tr], oa_tr=oa_target[tr])
                yp_all[te] = r["y_pred"]
            ok = yp_all >= 0
            rows.append(dict(model=m_name, subset=label, n=int(ok.sum()), qwk=ORD.qwk(y_idx[ok], yp_all[ok], K)))
    sens = pd.DataFrame(rows)
    display(sens.pivot(index="model", columns="subset", values="qwk").round(3))
    sens.to_csv(OUT_DIR / "sensitivity_high_conf.csv", index=False)

## 7) Lưu kết quả

In [ ]:
pooled_summary.to_csv(OUT_DIR / "cv_summary_pooled.csv")
fold_summary.to_csv(OUT_DIR / "cv_summary_folds.csv")
keep = ["case_id", "subject", "KL"] + (["source_dataset"] if "source_dataset" in df.columns else [])
oof_rows = df[keep].copy()
for (fs, m, s), st in OOF.items():
    if s != SEED_SHOW:
        continue
    oof_rows[f"pred_{fs}__{m}"] = [CLASSES[i] if i >= 0 else np.nan for i in st["y_pred"]]
    if not np.isnan(st["p_thr"]).all():
        for k in range(K - 1):
            oof_rows[f"pthr_{fs}__{m}__gt{CLASSES[k]}"] = st["p_thr"][:, k]
oof_rows.to_csv(OUT_DIR / f"oof_predictions_seed{SEED_SHOW}.csv", index=False)
json.dump(dict(source=str(SRC), classes=CLASSES, n=len(df), n_splits=N_SPLITS, seeds=SEEDS, xgb=XGB_KW,
               mlp_epochs=MLP_EPOCHS, decode="count: y = #{k : p_k > 0.5}", oa_target="KL>=2",
               feature_sets={k: len(v) for k, v in FEATURE_SETS.items()}),
          open(OUT_DIR / "run_config.json", "w"), indent=2)
print("da luu vao", OUT_DIR)

## Ghi chú
- Đầu ra chính thức của mọi model ordinal là **đếm ngưỡng**; E còn có head softmax để đo mức mâu thuẫn giữa các head —
  báo cáo phải chọn một và ghi rõ.
- `L_mono` là phạt mềm, `monotonic_violation_rate` cho biết còn bao nhiêu hàng vi phạm. Muốn đơn điệu theo cấu trúc: CORAL
  (bias có thứ tự, chung trọng số) hoặc CORN (xác suất có điều kiện).
- `L_OA` trùng với ngưỡng p₁ nên E ≈ D + tăng trọng số một ngưỡng + head softmax. Nếu E không hơn D thì phần "multi-task" không
  đóng góp gì trên feature bảng.
- QWK một split có sd ~0.05–0.1; kết luận "ordinal tốt hơn" cần chênh vượt sd qua seed, và nên kiểm bằng paired bootstrap
  trên dự đoán out-of-fold (`bsc.metrics.paired_bootstrap`).